# 1/3 — Dataset prep + EDA (Kaggle, no GPU needed for this notebook)

Companion to `docs/FINE_TUNING_YOLO_GUIDE.md`. This is step 1 of 3 — deliberately split from training (`02_train.ipynb`) so you get a reviewable class-distribution report *before* spending any GPU time, and so a training-session cutoff never means re-downloading LVIS (~20.7GB) from scratch.

**What this notebook produces** (as this notebook's committed Output, which `02_train.ipynb` attaches as an input dataset — see the last cell):
- `lostfound/` — the merged, remapped 48-class training set
- `coco_forgetting_check/` — the automatic catastrophic-forgetting eval set
- `eda_report/` — per-class instance counts, an imbalance flag, and a chart

Run top to bottom. GPU accelerator is NOT required for this notebook (CPU is fine — nothing here trains a model).

In [ ]:
!echo "--- /kaggle/input contents ---"; ls -la /kaggle/input/ 2>&1; ls -la /kaggle/input/*/ 2>&1

In [ ]:
!pip install -q ultralytics==8.4.106
import ultralytics
ultralytics.checks()

## 1. Copy the scripts into place (cheap — do this before the expensive download)

Upload `finetuning/prepare_dataset.py`, `finetuning/prepare_coco_forgetting_check.py`, and `finetuning/eda.py` from this repo as a Kaggle Dataset (or **Add Data → Upload**) and attach it to this notebook. Keeps 20 of COCO's 80 classes and adds 28 new LVIS-backed classes — see `prepare_dataset.py`'s module docstring or `docs/FINE_TUNING_YOLO_GUIDE.md` Section 2 for the full rationale.

**Add Data attaches an uploaded dataset read-only under `/kaggle/input/datasets/<owner>/<slug>/`** when wired via `kernel-metadata.json`'s `dataset_sources` (API-triggered `kaggle kernels push`) — confirmed directly via the diagnostic cell above; this is a different, deeper path than the flatter `/kaggle/input/<slug>/` some older guidance assumes. It doesn't land in `/kaggle/working/` (this notebook's cwd) either way, so the next cell copies the three scripts into place before anything tries to run them — deliberately placed **before** the download cells below: a broken script attachment then fails in seconds, not after a ~20-minute COCO download. If you're using this repo's `kaggle kernels push` setup (`finetuning/kaggle/01-dataset-prep-eda/kernel-metadata.json`), the scripts dataset is already wired via `dataset_sources` — no manual Add Data click needed, `SCRIPTS_INPUT_DIR` below is already pointed at its real mounted path.

In [ ]:
import shutil, glob

SCRIPTS_INPUT_DIR = "/kaggle/input/datasets/dceshubh/vision-assist-finetuning-scripts"  # API-attached dataset_sources mount at /kaggle/input/datasets/<owner>/<slug>/, NOT /kaggle/input/<slug>/ like manual Add Data — confirmed via the diagnostic cell above

# Add Data attaches datasets read-only under /kaggle/input/ — never into /kaggle/working/,
# which is this notebook's cwd for the !python cells below. Copy the scripts into place
# first, or every !python call below fails with "No such file or directory".
for f in glob.glob(f"{SCRIPTS_INPUT_DIR}/*.py"):
    shutil.copy(f, ".")
copied = glob.glob("*.py")
assert {"prepare_dataset.py", "prepare_coco_forgetting_check.py", "eda.py"} <= set(copied), f"Missing expected script(s) — only found {copied}. Check SCRIPTS_INPUT_DIR above."
print("Copied:", copied)

## 2. Force Ultralytics to download + convert COCO, then LVIS

**COCO first, LVIS second — not the other way around, and not independently.** LVIS reuses COCO's own `train2017`/`val2017` JPEGs, but Ultralytics' `lvis.yaml` doesn't know that — its own `download:` block fetches `train2017.zip`(19G)+`val2017.zip`(1G) directly into `datasets/lvis/images/`, a **second, separate copy** of the exact same ~20GB of images `coco.yaml` already downloads into `datasets/coco/images/`. Triggering both independently (the old order) silently doubles the download and is why this step can blow past Kaggle's working-disk limit even though the final dataset is nowhere near that big. The next three cells: download COCO's images once, symlink LVIS's expected image directory to that same copy, then trigger LVIS — `safe_download()` only re-fetches a zip if it's missing at that exact path, so through the symlink it finds COCO's zip already there and skips straight to LVIS's own much smaller `lvis-labels-segments.zip`.

**Also redirect `datasets_dir` away from `/kaggle/working` first (next cell).** Ultralytics defaults `datasets_dir` to `<cwd>/datasets` when it finds no git repo — on Kaggle that resolves to `/kaggle/working/datasets`, the ~20GB persisted-Output volume, not the much larger ephemeral scratch space. That's what actually throws "Insufficient free disk space" here, independent of the LVIS/COCO duplication above — even a single ~20GB COCO download doesn't reliably fit in `/kaggle/working`'s quota. `/kaggle/tmp` is the standard Kaggle scratch path for exactly this (not persisted after the session, which is fine — only the small, curated `lostfound/`/`coco_forgetting_check/`/`eda_report/` outputs below need to survive as this notebook's committed Output, not the raw LVIS/COCO source data). **A plain `settings.update(datasets_dir=...)` is not enough** — cell 1 above already imported `ultralytics`, which already froze `DATASETS_DIR` as a plain constant in both `ultralytics.utils` and `ultralytics.data.utils` (each via its own `from ultralytics.utils import DATASETS_DIR`); updating the settings dict afterward doesn't reach either already-bound copy, so the actual download code keeps using the stale `/kaggle/working/datasets` path even though `settings["datasets_dir"]` correctly reports the new value — confirmed by an actual run that still errored identically despite the update. The next cell patches `settings.json` on disk directly, then force-reimports every `ultralytics.*` module so `DATASETS_DIR` gets recomputed everywhere it was bound, with an assertion that fails immediately if that didn't work — rather than discovering it two minutes into a download attempt.

**Triggering the download: `check_det_dataset()` directly, not `model.train()`.** An earlier version of this notebook called `model.train(data="coco.yaml", epochs=1)` with a comment saying "interrupt after epoch 1 starts" — fine for a human watching, but an unattended `kaggle kernels push` run has nobody to interrupt it: confirmed live, it silently trained one full epoch over all 118,287 COCO `train2017` images on CPU (~8.3s/iteration, ~17+ hours projected) before this was caught. `check_det_dataset()` is the exact function `model.train()` calls internally to download+verify a dataset, *before* it ever builds a model or starts training — calling it directly gets the same download with zero training compute, deterministically, with nothing to interrupt.

**The symlink has a side effect: it also blocks LVIS's own labels from unzipping.** Confirmed live — `check_det_dataset("lvis.yaml")` downloads `lvis-labels-segments.zip` fine, but Ultralytics' `unzip_file()` treats *any* pre-existing, non-empty target directory as "already fully extracted" and skips unzipping unconditionally (`exist_ok=False` by default) — it can't tell that the only reason `lvis/` is non-empty is our own `images/` symlink, not real extracted labels. Net effect: `find_candidates()` found 0 LVIS images on the run that first hit this, because the labels simply were never on disk. The cell two below forces the extraction (`exist_ok=True`) right after the LVIS trigger.

In [ ]:
import json, sys
from pathlib import Path

SCRATCH_DATASETS_DIR = "/kaggle/tmp/datasets"  # large ephemeral scratch, NOT the ~20GB-capped /kaggle/working
Path(SCRATCH_DATASETS_DIR).mkdir(parents=True, exist_ok=True)

# cell 1's `import ultralytics` already ran, which already created and loaded
# settings.json AND already bound `DATASETS_DIR = Path(SETTINGS["datasets_dir"])` as a
# frozen constant inside ultralytics.utils (and a SEPARATE copy inside ultralytics.data.utils,
# via its own `from ultralytics.utils import DATASETS_DIR`). settings.update(datasets_dir=...)
# only rewrites the persisted settings.json file and the live settings dict — it does NOT
# reassign either already-bound DATASETS_DIR name, so the actual download code still used
# the stale /kaggle/working/datasets path despite settings["datasets_dir"] correctly showing
# the new value (confirmed by a real failed run). Patch the file directly, then force every
# ultralytics.* module to re-import from scratch so DATASETS_DIR gets recomputed everywhere
# it was bound, not just in the settings dict.
from ultralytics.utils import SETTINGS_FILE
settings_data = json.loads(Path(SETTINGS_FILE).read_text())
settings_data["datasets_dir"] = SCRATCH_DATASETS_DIR
Path(SETTINGS_FILE).write_text(json.dumps(settings_data, indent=4))

for mod_name in [m for m in sys.modules if m == "ultralytics" or m.startswith("ultralytics.")]:
    del sys.modules[mod_name]

from ultralytics import YOLO, settings
from ultralytics.data.utils import DATASETS_DIR as DATA_UTILS_DATASETS_DIR
print("settings['datasets_dir']:", settings["datasets_dir"])
print("ultralytics.data.utils.DATASETS_DIR:", DATA_UTILS_DATASETS_DIR)
assert str(DATA_UTILS_DATASETS_DIR) == SCRATCH_DATASETS_DIR, (
    f"Re-import didn't pick up the new datasets_dir ({DATA_UTILS_DATASETS_DIR} != {SCRATCH_DATASETS_DIR}) "
    "- downloads would still land in /kaggle/working and hit the same disk error."
)

!df -h /kaggle/tmp /kaggle/working

In [ ]:
from ultralytics import settings
from ultralytics.data.utils import check_det_dataset
print("datasets_dir:", settings["datasets_dir"])

# check_det_dataset() is what model.train() calls internally to download+verify a
# dataset BEFORE it ever builds a model or enters a training loop — calling it directly
# triggers the exact same download with zero training compute. The old approach here
# was model.train(data="coco.yaml", epochs=1, ...) with a comment saying "interrupt
# after epoch 1 starts": that's fine for a human watching, but an unattended
# `kaggle kernels push` run has nobody to interrupt it, so it silently trained one full
# epoch over all 118,287 COCO train2017 images on CPU (confirmed live: ~8.3s/iteration,
# ~17+ hours projected for just one epoch) before this was caught and fixed.
check_det_dataset("coco.yaml")

In [ ]:
import os
from pathlib import Path

datasets_dir = Path(settings["datasets_dir"])
lvis_images = datasets_dir / "lvis" / "images"
lvis_images.parent.mkdir(parents=True, exist_ok=True)
if not lvis_images.exists():
    os.symlink(datasets_dir / "coco" / "images", lvis_images)
print("lvis/images ->", lvis_images.resolve() if lvis_images.is_symlink() else "already a real directory (not symlinked, left as-is)")

In [ ]:
from ultralytics.data.utils import check_det_dataset

# with images/ symlinked above, this should only fetch lvis-labels-segments.zip, not
# re-download train2017/val2017 — see the no-training rationale in the cell above
check_det_dataset("lvis.yaml")

In [ ]:
from ultralytics.utils.downloads import unzip_file

# check_det_dataset("lvis.yaml") just downloaded lvis-labels-segments.zip but — confirmed
# directly in the live log — SKIPPED unzipping it, warning "destination directory
# .../lvis is not empty". That is a side effect of the symlink two cells above: Ultralytics'
# unzip_file() treats ANY pre-existing, non-empty target directory as "already fully
# extracted" and skips unconditionally (exist_ok=False by default) — it has no way to know
# the only reason lvis/ is non-empty is OUR OWN images/ symlink, not real extracted labels.
# This is exactly why find_candidates() found 0 LVIS images on the prior run: the labels
# zip was downloaded but its contents were never actually unzipped onto disk. Force it,
# bypassing that same coarse check this time (exist_ok=True) — this is the identical call
# safe_download() already made internally, just with the one flag flipped.
labels_zip = datasets_dir / "lvis-labels-segments.zip"
assert labels_zip.exists(), f"{labels_zip} not found — did the LVIS trigger cell above actually download it?"
unzip_file(file=labels_zip, path=datasets_dir, exist_ok=True)
lvis_labels_dir = datasets_dir / "lvis" / "labels" / "train2017"
n_label_files = len(list(lvis_labels_dir.glob("*.txt"))) if lvis_labels_dir.exists() else 0
print(f"Force-extracted {labels_zip} — {n_label_files} label files now at {lvis_labels_dir}")
assert n_label_files > 0, f"Still no label files at {lvis_labels_dir} after force-extraction — something else is wrong."

In [ ]:
from pathlib import Path

# Ultralytics' download()/safe_download() default to delete=False, and neither coco.yaml
# nor lvis.yaml's own download: block overrides that — confirmed in their source on
# GitHub. So both zips are left on disk after extraction: on top of the ~19-20GB of
# already-unzipped jpgs, that's roughly 40GB total in /kaggle/tmp, not the ~20GB this
# notebook's own markdown estimates. Harmless while /kaggle/tmp has room (confirmed
# ~967GB free on this container), but delete them anyway rather than silently assume
# that margin always holds — leaving ~20GB of already-extracted zips around is pure
# waste once both datasets are verified, and could resurrect the exact disk error this
# notebook already spent several attempts fixing if /kaggle/tmp's real quota is ever
# tighter than it was here.
for zip_name in ("coco/images/train2017.zip", "coco/images/val2017.zip"):
    zip_path = datasets_dir / zip_name
    if zip_path.exists():
        size_gb = zip_path.stat().st_size / 1e9
        zip_path.unlink()
        print(f"Deleted {zip_path} ({size_gb:.1f}GB)")

In [ ]:
import time
from pathlib import Path

# Ground-truth check, run directly in this kernel (no subprocess, so no buffering
# ambiguity) before handing off to prepare_dataset.py. A prior run's COCO label scan
# sat with zero new output for 45+ minutes at the exact same step LVIS's scan (114,262
# files) completed quickly on — ruled out generic small-file I/O slowness via a separate
# throwaway benchmark kernel (120K synthetic files glob+read in ~4s on this same Kaggle
# infra), so the remaining open question is simply: how many files are actually in
# COCO's labels/train2017, and how long does listing them really take here.
coco_labels_dir = Path(settings["datasets_dir"]) / "coco" / "labels" / "train2017"
t0 = time.time()
exists = coco_labels_dir.exists()
n_txt = len(list(coco_labels_dir.glob("*.txt"))) if exists else -1
print(f"COCO labels dir: {coco_labels_dir}")
print(f"  exists: {exists}")
print(f"  .txt file count: {n_txt}")
print(f"  glob took: {time.time() - t0:.2f}s")
if exists:
    all_entries = list(coco_labels_dir.iterdir())
    print(f"  total entries (any type): {len(all_entries)}")
    non_txt = [p for p in all_entries if p.suffix.lower() != ".txt"]
    print(f"  non-.txt entries: {len(non_txt)}" + (f" (sample: {non_txt[:5]})" if non_txt else ""))


## 3. Build the merged 48-class training dataset

Runs `prepare_dataset.py` against the COCO/LVIS data downloaded above, filtering/remapping into this project's 48-class scheme and writing `lostfound.yaml`.

In [ ]:
DATASETS_DIR = settings["datasets_dir"]  # now /kaggle/tmp/datasets, set by the redirect cell above

# SWEEP VARIANT: --coco-sample-size raised 750 -> 5000 (~6.7x). The baseline run's
# forgetting-check showed scissors/toothbrush (the 2 kept classes with by far the
# fewest training instances, 180/143) lost 66% of their mAP50-95 relative to base
# YOLO - merge_and_sample() draws this many COCO-only images UNIFORMLY AT RANDOM from
# the ~64,696-image pool of "has a kept class, no novel class" candidates, so a bigger
# draw proportionally pulls in more of the rare kept classes too (not a smarter/
# stratified sample, just a bigger one) - see docs/FINE_TUNING_YOLO_GUIDE.md and
# finetuning/kaggle/RUN_LOG.md's hyperparameter sweep section for the full reasoning.
!python -u prepare_dataset.py \
    --lvis-root {DATASETS_DIR}/lvis \
    --coco-root {DATASETS_DIR}/coco \
    --output-dir /kaggle/working/lostfound \
    --coco-sample-size 5000

assert _exit_code == 0, (
    f"prepare_dataset.py exited with code {_exit_code} — stopping here rather than "
    "letting later cells (EDA, notebook 2's training) silently run against missing/"
    "partial output. A bare !python call's exit code is NOT a cell error in Jupyter/"
    "Kaggle by default, so without this check a failure here would have gone unnoticed."
)

## 4. Build the forgetting-check set (fully automatic)

Unlike the golden set (which needs real self-captured photos, done separately — see `finetuning/README.md`), this one needs no new annotation: it's a filtered/remapped sample of COCO's own `val2017` ground truth.

In [ ]:
!python -u prepare_coco_forgetting_check.py \
    --coco-root {DATASETS_DIR}/coco \
    --output-dir /kaggle/working/coco_forgetting_check \
    --sample-size 75

assert _exit_code == 0, (
    f"prepare_coco_forgetting_check.py exited with code {_exit_code}. This script's own "
    "`raise SystemExit(...)` on zero candidates is a nonzero shell exit, not a Jupyter "
    "cell error — nothing downstream in THIS notebook ever reads coco_forgetting_check/, "
    "so a silent failure here would only surface later in 03_evaluate_finetune_checks.ipynb, "
    "after GPU time has already been spent training in 02_train.ipynb."
)

## 5. EDA — per-class instance counts and imbalance check

**Read this before moving on.** `prepare_dataset.py` only ever printed how many *images* matched each source — never how many box *instances* per class actually survived the merge. A class can show up in plenty of images but still have very few boxes. `eda.py` counts real per-class instances (train+val), flags anything under the imbalance floor (default 50), and writes a CSV + chart you can actually look at.

In [ ]:
!python -u eda.py --dataset-dir /kaggle/working/lostfound --output-dir /kaggle/working/eda_report --imbalance-floor 50

assert _exit_code == 0, f"eda.py exited with code {_exit_code} — see its output above for why."

In [ ]:
from IPython.display import Image, display
import json

display(Image(filename="/kaggle/working/eda_report/class_distribution.png"))
print(json.dumps(json.load(open("/kaggle/working/eda_report/eda_summary.json")), indent=2))

## 6. Review gate — decide before training, not after

Look at `eda_report/class_distribution.csv` and the chart above:
- **Zero-instance classes** mean training will never learn them at all — fix the dataset before training, don't discover this from a confusing eval result later.
- **Classes under the imbalance floor** are candidates for a Roboflow Universe fallback (`docs/FINE_TUNING_YOLO_GUIDE.md` Section 3.2) — check per-class, since LVIS coverage varies a lot by class.
- A large `imbalance_ratio_max_over_min` is expected (LVIS classes aren't uniformly common) — the question is whether any *individual* class is too sparse to learn anything, not whether the distribution is perfectly even.

## 7. Persist this notebook's output for 02_train.ipynb

**File → Save Version** now (even without re-running) — this commits `/kaggle/working` (i.e. `lostfound/`, `coco_forgetting_check/`, `eda_report/`) as this notebook's Output, which becomes a Kaggle Dataset. In `02_train.ipynb`, **Add Data** → this notebook's output version, instead of re-downloading or re-merging anything there.